In [ ]:
repository_filter: list[str] = []
coupling_threshold: str = "10"
cohesion_threshold: str = "0.5"

In [ ]:
from code_data_science import data_table as dt
from moderne_visualizations_misc.reusable import quality_utils as qu
import plotly.graph_objects as go
import warnings

warnings.simplefilter("ignore")

cbo_thresh = float(coupling_threshold)
tcc_thresh = float(cohesion_threshold)

df = dt.read_csv("../samples/class_quality_metrics.csv")
df = qu.filter_repos(df, repository_filter)
df = qu.add_repo_short(df)
df = qu.add_class_short(df)

In [ ]:
if len(df) == 0:
    fig = qu.empty_figure()
else:
    fig = go.Figure()

    # Quadrant labels
    labels = [
        (cbo_thresh / 2, tcc_thresh / 2, "Island", "#90CAF9"),
        (cbo_thresh / 2, tcc_thresh + (1 - tcc_thresh) / 2, "Healthy", "#A5D6A7"),
        (cbo_thresh * 1.5, tcc_thresh / 2, "Spaghetti", "#EF9A9A"),
        (cbo_thresh * 1.5, tcc_thresh + (1 - tcc_thresh) / 2, "Hub", "#FFE082"),
    ]
    for lx, ly, label, color in labels:
        fig.add_annotation(
            x=lx, y=min(ly, 0.95),
            text=f"<b>{label}</b>",
            showarrow=False,
            font=dict(size=14, color=color),
            opacity=0.7,
        )

    # Quadrant divider lines
    fig.add_hline(y=tcc_thresh, line_dash="dash", line_color="#888", opacity=0.5)
    fig.add_vline(x=cbo_thresh, line_dash="dash", line_color="#888", opacity=0.5)

    # Scatter plot
    wmc_min = df["wmc"].min()
    wmc_range = df["wmc"].max() - wmc_min if df["wmc"].max() > wmc_min else 1
    sizes = 8 + 30 * (df["wmc"] - wmc_min) / wmc_range

    fig.add_trace(
        go.Scatter(
            x=df["cbo"],
            y=df["tcc"],
            mode="markers",
            marker=dict(
                size=sizes,
                color=df["maintainabilityIndex"],
                colorscale=qu.HEALTH_COLORSCALE,
                cmin=0,
                cmax=100,
                colorbar=dict(title="Maintainability<br>Index"),
                line=dict(width=0.5, color="#ddd"),
            ),
            text=df["className"],
            customdata=df[["wmc", "repoShort", "lcom4"]].values,
            hovertemplate=(
                "<b>%{text}</b><br>"
                "Repo: %{customdata[1]}<br>"
                "CBO (coupling): %{x}<br>"
                "TCC (cohesion): %{y:.2f}<br>"
                "WMC: %{customdata[0]}<br>"
                "LCOM4: %{customdata[2]}<br>"
                "Maintainability: %{marker.color:.0f}"
                "<extra></extra>"
            ),
        )
    )

    x_max = max(df["cbo"].max() * 1.1, cbo_thresh * 2)
    fig.update_layout(
        title="Coupling vs Cohesion Quadrant",
        xaxis=dict(title="CBO (Coupling Between Objects)", range=[0, x_max]),
        yaxis=dict(title="TCC (Tight Class Cohesion)", range=[-0.05, 1.05]),
        height=700,
        width=900,
        plot_bgcolor="white",
        showlegend=False,
        margin=dict(l=60, r=60, t=60, b=60),
    )
fig.show()